# StayIQ — Module 3: Personalized Recommendation Engine

This notebook demonstrates how to design a hybrid recommender engine for hotels. It integrates Collaborative Filtering (via Matrix Factorization) to recommend hotels matching historic user rating patterns, and Content-Based Filtering (via TF-IDF amenity indexing) to match metadata queries.

### Pipeline Steps:
1. **Hotel Content Profiling**: Vectorize hotel amenities using TF-IDF.
2. **Content Similarity Matrix**: Compute cosine similarity across amenities profiles.
3. **Collaborative Filtering**: Implement a matrix factorization routine (SVD / low-rank approximation) on user rating matrices.
4. **Hybrid Scoring Engine**: Combine collaborative ratings forecasts with content similarity indicators.
5. **Inference pipeline**: Query recommendations for user filters.

In [ ]:
# 1. Imports and setup
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# 2. Mock Hotel Database Setup
hotels_data = {
    'hotel_id': ['hot_001', 'hot_002', 'hot_003', 'hot_004', 'hot_005', 'hot_006'],
    'hotel_name': ['Hotel Altis Prime', 'Lisbon Plaza Hotel', 'The Yeatman Palace', 'Porto Bay Flores', 'Algarve Luxury Beach Resort', 'Tivoli Carvoeiro Hotel'],
    'location': ['Lisbon', 'Lisbon', 'Porto', 'Porto', 'Albufeira', 'Carvoeiro'],
    'price_per_night': [120.0, 95.0, 250.0, 180.0, 160.0, 220.0],
    'rating': [4.6, 4.4, 4.9, 4.7, 4.5, 4.8],
    'amenities': [
        'pool wifi parking spa gym',
        'wifi parking breakfast bar',
        'pool wifi parking spa gym restaurant view',
        'wifi spa restaurant breakfast gym',
        'pool wifi beach parking restaurant',
        'pool wifi beach spa bar restaurant'
    ]
}
df_hotels = pd.DataFrame(hotels_data)
df_hotels

In [ ]:
# 3. Content-Based Filtering (TF-IDF on Amenities)
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_hotels['amenities'])

# Compute pairwise cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
df_similarity = pd.DataFrame(cosine_sim, index=df_hotels['hotel_name'], columns=df_hotels['hotel_name'])
print("Amenities Cosine Similarity Matrix:")
df_similarity

In [ ]:
# 4. Collaborative Filtering Mock (User Ratings Matrix)
# Imagine we have ratings from 5 users for these 6 hotels
ratings_data = np.array([
    [5.0, 4.0, np.nan, np.nan, 3.0, np.nan],
    [np.nan, 5.0, 5.0, 4.0, np.nan, np.nan],
    [3.0, np.nan, np.nan, 5.0, 4.0, 5.0],
    [4.0, 4.0, np.nan, np.nan, np.nan, 4.0],
    [np.nan, np.nan, 5.0, 5.0, 5.0, np.nan]
])

# Fill missing values with user mean for SVD approximation
user_means = np.nanmean(ratings_data, axis=1, keepdims=True)
ratings_filled = np.where(np.isnan(ratings_data), user_means, ratings_data)

# Run Singular Value Decomposition (SVD)
U, sigma, Vt = np.linalg.svd(ratings_filled - user_means, full_matrices=False)
# We keep k=2 latent factors
k = 2
sigma_k = np.diag(sigma[:k])
ratings_reconstructed = np.dot(U[:, :k], np.dot(sigma_k, Vt[:k, :])) + user_means

df_reconstructed = pd.DataFrame(ratings_reconstructed, columns=df_hotels['hotel_name'])
print("SVD Reconstructed Predicted User Ratings:")
df_reconstructed

In [ ]:
# 5. Hybrid Recommendation Query
# Compute recommendations for User 0 who likes pool/parking and has Lisbon location constraint
user_id = 0
preferred_location = 'Lisbon'

# Filter hotels by location first
matching_idx = df_hotels[df_hotels['location'] == preferred_location].index.tolist()

recommendations = []
for idx in matching_idx:
    hotel = df_hotels.iloc[idx]
    collaborative_score = ratings_reconstructed[user_id, idx] / 5.0 # normalize to 0-1
    rating_score = (hotel['rating'] - 3.0) / 2.0  # normalize 3-5 rating to 0-1
    
    # Combine factors
    hybrid_score = (0.5 * collaborative_score) + (0.5 * rating_score)
    recommendations.append({
        'hotel_name': hotel['hotel_name'],
        'hybrid_match_score': hybrid_score
    })
    
df_recs = pd.DataFrame(recommendations).sort_values(by='hybrid_match_score', ascending=False)
print("Final hybrid recommendations:")
df_recs